# Sentr - layer 2 training (Colab free GPU)

Fine-tunes the catalogue-shaped prompt-injection classifier that sits behind the rule layer.

**Defence only.** Nothing here composes, mutates or optimises an attack payload. Payload strings were fixed on Day 3 in `data/fixtures/`, copied verbatim from published research corpora or written by the project and labelled as such. This notebook only fits a detector to them.

**The held-out set is not here.** `notebooks/pack_for_colab.py` ships `train.jsonl` and `val.jsonl` only, and refuses to build an archive containing `test.jsonl`. The held-out set is opened once, on Day 5, on the local machine.

**Why Colab.** The development machine has 5.86 GB of RAM and measured 0.7 training windows/sec on CPU, which is about 2.4 hours per epoch. The same job here takes a few minutes.

### Run order
1. `Runtime -> Change runtime type -> T4 GPU`
2. Run every cell top to bottom
3. Upload `notebooks/sentr_colab.zip` when cell 3 asks (build it locally first: `python notebooks/pack_for_colab.py`)
4. The last cell downloads `sentr-classifier.zip` - unzip it into `models/sentr-classifier/` in the repo

In [ ]:
# 1. Confirm a GPU is attached. If this prints nothing, the runtime is CPU-only:
#    Runtime -> Change runtime type -> T4 GPU. Training still works, just slowly.
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

In [ ]:
# 2. Dependencies. Colab ships torch; these are the two the tokenizer needs
#    plus a transformers new enough for DebertaV2TokenizerFast offsets.
!pip -q install -U "transformers>=4.44" sentencepiece protobuf

In [ ]:
# 3. Upload the bundle built by notebooks/pack_for_colab.py.
#    (Alternative, if the data is committed and the repo is public:
#       !git clone https://github.com/rushii23-dev/Project-Sentr.git sentr_colab )
import os, zipfile
from google.colab import files

up = files.upload()
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    assert not any('test.jsonl' in n for n in z.namelist()), \
        'held-out set must never reach the training machine'
    z.extractall('.')
os.chdir('sentr_colab')
print(os.getcwd())
!ls -la sentr data/processed

In [ ]:
# 4. Train. Embeddings stay warm here - the memory pressure that forces them
#    frozen locally does not exist on a T4.
!python -m sentr.train_classifier \
    --epochs 3 \
    --base-model microsoft/deberta-v3-xsmall \
    --no-freeze-embeddings \
    --out models/sentr-classifier

In [ ]:
# 5. What the run actually produced: the val threshold sweep, and the two
#    thresholds calibrated from it. Read this before trusting the weights.
import json
meta = json.load(open('models/sentr-classifier/training_meta.json'))
print('base       ', meta['base_model'], '|', meta['parameters_total_m'], 'M params')
print('windows    ', meta['train_windows'], meta['window_label_counts'],
      '| dropped', meta['windows_dropped_partial_span'])
print('minutes    ', meta['minutes'])
print('thresholds ', meta['thresholds'])
print()
print(f"{'thresh':>8} {'recall%':>9} {'benign':>7} {'fpr%':>7}")
for r in meta['val_threshold_sweep']:
    if round(r['threshold'] * 100) % 5 == 0 or r['benign_hits'] == 0:
        print(f"{r['threshold']:>8} {r['recall_pct']:>9} {r['benign_hits']:>7} {r['fpr_pct']:>7}")

In [ ]:
# 6. Download the weights. Unzip into models/sentr-classifier/ in the repo,
#    then locally:  python eval/evaluate.py --split both
!cd models && zip -qr /content/sentr-classifier.zip sentr-classifier
from google.colab import files
files.download('/content/sentr-classifier.zip')

## After the download

```bash
# from the repo root
mkdir -p models && unzip -o ~/Downloads/sentr-classifier.zip -d models/
python eval/evaluate.py --split both
```

`models/` is gitignored - the weights are a build artefact, reproducible from this notebook. What gets committed is `eval/results/`, which is where the claims live.

If val recall is disappointing, that is the number that gets reported. Re-running this with different hyperparameters is fine; re-running it against `test.jsonl` is not.